In [269]:
import pandas as pd
import numpy as np


df = pd.read_csv("C:/Users/Youcode/Desktop/DEV Ai/DataStore360/data/RAW/store_data.csv", encoding="latin-1")
df_clean =  df.copy()




In [270]:
#using a standard format for all strings
id_cols = ["Row ID", "Order ID", "Customer ID", "Product ID"]
string_cols = df_clean.select_dtypes(include="object").columns.difference(id_cols)
for col in string_cols:
    df_clean[col] = df_clean[col].str.strip().str.lower()

In [271]:
#deleting duplicated

df_clean = df_clean.drop_duplicates()


In [272]:
#saving
df_clean.to_csv('../data/processed/store_data.csv')

In [273]:
#duplicated row id
df_clean["Row ID"].duplicated().sum()

df_clean['Row ID'] = range(1, len(df_clean)+1)

In [275]:
#fixing dates //coerce :if pandas cant interpret a value as date  it change it to Nat
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'], format='mixed', errors='coerce')

df_clean['Ship Date'] = pd.to_datetime(df_clean['Ship Date'] , format='mixed' , errors='coerce')


In [276]:
#fixing the ship days
df_clean['shipping days'] = (df_clean['Ship Date'] - df_clean['Order Date']).dt.days

invalid_dates = df_clean["Ship Date"] < df_clean["Order Date"]
df_valid_dates = df_clean[~invalid_dates].copy()

avg_shipping = df_valid_dates.groupby('Ship Mode')['shipping days'].mean().apply(np.floor).sort_values()

avg_shipping






df_clean[
    df_clean["Ship Mode"].isna() & invalid_dates
].groupby("Order ID")[
    ["Order ID", "Order Date", "Ship Date", "Ship Mode"]
].value_counts(dropna=False).sort_values(ascending=False)




Order ID        Order Date  Ship Date   Ship Mode
CA-2015-163181  2015-11-07  2010-01-01  NaN          1
CA-2017-110310  2017-10-27  2010-01-01  NaN          1
Name: count, dtype: int64

In [ ]:
#sorted orders with nan -- >300 found
pd.set_option('display.max_rows', None)
mask = df_clean["Ship Mode"].isna()

orders_with_nan = df_clean.loc[mask, "Order ID"].unique()

df_clean[
    df_clean["Order ID"].isin(orders_with_nan)
][["Order Date","Ship Date","Order ID", "Ship Mode"]].sort_values("Order ID")

In [280]:
#calculating the mode of order ship for each order to fill the missing order ship

order_ship_mode = df_clean.groupby('Order ID')['Ship Mode'].agg(lambda x : x.mode()[0] if not x.mode().empty else np.nan)


#filling using fillna and map that do : for each row take its order id and lookup the right shipping mode
df_clean['Ship Mode'] =df_clean['Ship Mode'].fillna(df_clean['Order ID'].map(order_ship_mode))

#this reduce orders with nan from >300 to 70

In [281]:
#the 70 remaining 
mask = df_clean['Ship Mode'].isna()
df_clean.loc[1509, ['Order ID', 'Order Date', 'Ship Date', 'shipping days','Ship Mode']]

#creating a function that predict the ship mode based on  shipping days and shipping avg

def predict_ship_mode(days):
    diffs = (avg_shipping - days).abs()
    return diffs.idxmin()

predict_ship_mode(5)
#applying the function to the orders with ship mode nan
df_clean.loc[mask , 'Ship Mode'] = df_clean.loc[mask,'shipping days'].apply(predict_ship_mode)





In [282]:
#now fixing the rest invalid shiping dates based on their ship mode

df_clean[invalid_dates][["Order ID","Order Date","Ship Date",'Ship Mode']]


df_clean.loc[invalid_dates, 'Ship Date'] = (df_clean.loc[invalid_dates, 'Order Date'] + 
                                            pd.to_timedelta(
                                                df_clean.loc[invalid_dates, 'Ship Mode'].map(avg_shipping),
                                                unit="D"
                                                )
                                            )

# df_clean.groupby('Order ID').sum()

df_clean.loc[1562	, ['Order Date', 'Ship Date','Ship Mode']]



Order Date    2017-06-30 00:00:00
Ship Date     2017-06-30 00:00:00
Ship Mode                same day
Name: 1562, dtype: object

In [ ]:
#fixing typo on segment column
df_clean.Segment.unique()

segment_mapping = {
    'consumerr' : 'consumer',
    'home ofice' : 'home office',
    'corporrate' : 'corporate'
}

df_clean['Segment'] = df_clean['Segment'].replace(segment_mapping)

df_clean["Segment"].unique()

10064 10011


In [319]:
pd.set_option('display.max_rows', None)
df_clean.groupby(["State" , "City"])[
    ["State", "City", "Postal Code"]
].value_counts(dropna=False).sort_values(ascending=False)




State                 City               Postal Code
new york              new york city      10035.0        258
                                         10009.0        227
                                         10024.0        223
california            san francisco      94122.0        193
new york              new york city      10011.0        188
california            san francisco      94110.0        162
washington            seattle            98105.0        162
pennsylvania          philadelphia       19134.0        156
california            los angeles        90049.0        149
washington            seattle            98103.0        148
pennsylvania          philadelphia       19140.0        146
california            san francisco      94109.0        138
                      los angeles        90045.0        137
                                         90036.0        119
ohio                  columbus           43229.0        118
california            los angeles        90004.